In [1]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys
import ctypes

try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"
ARTIFACTS_DIR = f"{codebase_path}/artifacts"
print("💻 Local Lab Server Environment Loaded.")


💻 Local Lab Server Environment Loaded.


In [2]:
# === CONFIGURATION ===
USE_UNSLOTH = False  # Granite typically uses standard Transformers + PEFT
MODEL_ID = "ibm-granite/granite-3.1-2b-instruct"
PROMPT_TYPE = "fullInfo"  # "structOnly" or "fullInfo"
SFT_METHOD = "LoRA"  # e.g., "DoRA", "LoRA", or "None"
max_seq_length = 1500
load_in_4bit = True
RUN_VALIDATION = False  # Set to True to evaluate on validation set instead of test set

# Construct the correct directory dynamically
safe_model_name = MODEL_ID.split('/')[-1]
BEST_MODEL_DIR = f"{MODELS_DIR}/{safe_model_name}_{PROMPT_TYPE}_{SFT_METHOD}"
print(f"Loading model: {BEST_MODEL_DIR}")

Loading model: ..//output/models/granite-3.1-2b-instruct_fullInfo_LoRA


## 1. Load Data & Base Model

In [3]:
# === IMPORT LIBRARIES & CONDITIONAL UNSLOTH SETUP ===
import json
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer
from src.utils.prompts import format_prompt
import re

# Conditionally import Unsloth based on USE_UNSLOTH config option
if USE_UNSLOTH:
    try:
        from unsloth import FastLanguageModel
        print("✅ Unsloth library successfully imported!")
    except ImportError:
        print("⚠️ Unsloth NOT available! Falling back to standard Transformers + PEFT.")
        USE_UNSLOTH = False
else:
    print("ℹ️ Unsloth disabled by configuration. Using standard Transformers + PEFT.")

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

if RUN_VALIDATION:
    val_filename = "val_structural.jsonl" if PROMPT_TYPE == "structOnly" else "val_full_info.jsonl"
    data_path = f"{codebase_path}/output/cache/{val_filename}"
    test_data = load_jsonl(data_path)
    print(f"Loaded {len(test_data)} VALIDATION samples from {data_path}")
else:
    data_path = f"{codebase_path}/output/cache/test_filtered_top4_BAAI-bge-small-en-v1.5.jsonl"
    test_data = load_jsonl(data_path)
    print(f"Loaded {len(test_data)} TEST samples from {data_path}")

ℹ️ Unsloth disabled by configuration. Using standard Transformers + PEFT.
Loaded 1181 TEST samples from ..//output/cache/test_filtered_top4_BAAI-bge-small-en-v1.5.jsonl


In [4]:
# === LOAD MODEL & TOKENIZER ===
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Determine compute dtype
COMPUTE_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

if USE_UNSLOTH:
    print("Loading with Unsloth (4-bit)...")
    dtype = None  # Unsloth automatically handles dtype
    if SFT_METHOD == "None":
        print(f"Loading Base Model via Unsloth: {MODEL_ID}...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = MODEL_ID,
            max_seq_length = max_seq_length,
            dtype = dtype,
            load_in_4bit = load_in_4bit,
        )
    else:
        print(f"Loading SFT Model via Unsloth: {BEST_MODEL_DIR}...")
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name = BEST_MODEL_DIR,
                max_seq_length = max_seq_length,
                dtype = dtype,
                load_in_4bit = load_in_4bit,
            )
        except Exception as e:
            print(f"Could not load adapter {BEST_MODEL_DIR} via Unsloth. Falling back to Base Model: {MODEL_ID}...")
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name = MODEL_ID,
                max_seq_length = max_seq_length,
                dtype = dtype,
                load_in_4bit = load_in_4bit,
            )
    # Enable native 2x faster inference
    FastLanguageModel.for_inference(model)

else:
    print("Loading with standard Transformers + PEFT...")
    # Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Configure 4-bit Quantization if load_in_4bit is True
    if load_in_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
            bnb_4bit_use_double_quant=True
        )
    else:
        bnb_config = None

    # Load Model
    if SFT_METHOD == "None":
        print(f"Loading Base Model: {MODEL_ID}...")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=COMPUTE_DTYPE,
        )
    else:
        print(f"Loading Base Model & PEFT Adapter: {BEST_MODEL_DIR}...")
        try:
            from peft import PeftModel
            base_model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=COMPUTE_DTYPE,
            )
            model = PeftModel.from_pretrained(base_model, BEST_MODEL_DIR)
        except Exception as e:
            print(f"Could not load adapter {BEST_MODEL_DIR} via PEFT. Falling back to Base Model: {MODEL_ID}...")
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=COMPUTE_DTYPE,
            )

Loading with standard Transformers + PEFT...
Loading Base Model & PEFT Adapter: ..//output/models/granite-3.1-2b-instruct_fullInfo_LoRA...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## 2. Generate Predictions

In [5]:
def parse_letter(text):
    if "</think>" in text:
        text = text.split("</think>")[-1]
    elif "<think>" in text:
        return 'A'
    match = re.search(r'\b[A-H]\b', text)
    if match:
        return match.group(0)
    match = re.search(r'[A-H]', text)
    if match:
        return match.group(0)
    return 'A' # Fallback

predictions = []
correct = 0

for row in tqdm(test_data, desc="Generating Predictions"):
    messages = format_prompt(row, is_test=not RUN_VALIDATION)
    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except TypeError:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Pre-fill assistant response with a closed think block for reasoning models to bypass slow thinking phase
    try:
        vocab = tokenizer.get_vocab()
        if "<think>" in vocab:
            prompt += "<think>\nDone thinking.\n</think>\n"
    except Exception:
        pass
        
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    generation_kwargs = {
        "max_new_tokens": 10,
        "pad_token_id": tokenizer.eos_token_id,
        "do_sample": False
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_kwargs
        )
        
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    pred_letter = parse_letter(generated_text)
    
    result = {
        'id': row.get('id', len(predictions)),
        'answer': pred_letter
    }
    
    if RUN_VALIDATION and 'answer' in row:
        ground_truth = row['answer']
        result['ground_truth'] = ground_truth
        if pred_letter == ground_truth:
            correct += 1
            
    predictions.append(result)

if RUN_VALIDATION:
    accuracy = correct / len(predictions) if predictions else 0
    print(f"\n✅ Validation Accuracy: {accuracy:.2%} ({correct}/{len(predictions)})")

Generating Predictions:   0%|          | 0/1181 [00:00<?, ?it/s]/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Generating Predictions: 100%|██████████| 1181/1181 [02:33<00:00,  7.70it/s]


## 3. Save Submission

In [ ]:
if RUN_VALIDATION:
    print("Skipping CSV export because RUN_VALIDATION is True.")
else:
    model_dir_name = os.path.basename(BEST_MODEL_DIR)

    # If it's an SFT model, replace 'sft_' with 'submission_'. Otherwise just prefix it.
    if True:
        filename = f"submission_{model_dir_name}.csv"
    else:
        filename = f"submission_{model_dir_name}.csv"

    df_sub = pd.DataFrame(predictions)

    output_path = f"{codebase_path}/output/submissions/{filename}"

    print(output_path)

..//output/submissions/submission_granite-3.1-2b-instruct_fullInfo_LoRA_test.csv


In [7]:
if not RUN_VALIDATION:
    # Ensure output dir exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_sub.to_csv(output_path, index=False)
    print(f"Submission saved to {output_path}")

Submission saved to ..//output/submissions/submission_granite-3.1-2b-instruct_fullInfo_LoRA_test.csv


In [ ]:
# model_dir_name = os.path.basename(BEST_MODEL_DIR)

# # If it's an SFT model, replace 'sft_' with 'submission_'. Otherwise just prefix it.
if True:
    filename = f"submission_{model_dir_name}.csv"
# else:
#     filename = f"submission_{model_dir_name}.csv"

# df_sub = pd.DataFrame(predictions)

# output_path = f"{codebase_path}/output/submissions/{filename}"

# # Ensure output dir exists
# os.makedirs(os.path.dirname(output_path), exist_ok=True)
# df_sub.to_csv(output_path, index=False)
# print(f"Submission saved to {output_path}")